# FCSG-Net gate checks

Run this top to bottom before any long training run. Roughly two minutes of GPU.
**Do not proceed past a failing cell** — fix it locally, push, re-run the clone
cell, and start again. Every check that only ever runs on a laptop is a check
that is not running.

In [ ]:
# Clone once, pull on every later run. Editing code here is a trap: the next
# session starts from a fresh container and your edits are gone. Edit locally,
# push, re-run this cell.
import os, subprocess, sys
REPO_URL = "https://github.com/whynotramaa/fcsg-capstone"
REPO = "/kaggle/working/fcsg-capstone"
if os.path.isdir(REPO):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Attach a public DIV2K dataset in the sidebar (Add Input -> search "DIV2K"),
# then run this. Nothing hardcodes a slug, so any DIV2K dataset layout works as
# long as it contains DIV2K_train_HR and DIV2K_valid_HR directories.
sys.path[:0] = ["/kaggle/working/fcsg-capstone", "/kaggle/working/fcsg-capstone/src"]
from fcsg_net.utils import resolve_div2k
DATA = "/kaggle/input"
train_dir, val_dir = resolve_div2k(DATA)
print("train:", train_dir)
print("valid:", val_dir)

## 1. Does the data pipeline produce sane crops?

Look at the image. This is the step with no substitute: crops that are blank,
rotated wrong, or channel-swapped show up here and nowhere else.

In [ ]:
!python src/fcsg_net/data.py --data $train_dir --out /kaggle/working/data_check.png
from IPython.display import Image as Show
Show("/kaggle/working/data_check.png")

## 2. Is the model the size it should be?

In [ ]:
!python models/dncnn.py

## 3. Can it memorise one image?

200 steps on a single fixed crop with a single fixed noise realisation. The loss
must fall towards zero. A model that cannot do this has a bug, and finding it
here costs two minutes instead of ten hours.

In [ ]:
!rm -rf /kaggle/working/smoke
!python training/train.py --config configs/dncnn.toml --data /kaggle/input \
    --out /kaggle/working/smoke --steps 200 --batch 4 --overfit-one-image

## 4. Does resume actually resume?

Re-run the same command unchanged. It must print `RESUMED from ... at step 200`
rather than starting at 0. Kaggle kills sessions at 12 hours, so this cell is the
single most valuable one in the notebook.

In [ ]:
!python training/train.py --config configs/dncnn.toml --data /kaggle/input \
    --out /kaggle/working/smoke --steps 400 --batch 4 --overfit-one-image

## 5. Does it produce a measurable result?

Expect roughly 23-25 dB out from ~20 dB in after a few hundred steps, and output
that is visibly smoothed but still imperfect. That is the correct outcome: it
proves the chain executes and improves on the noisy input. The ~29 dB Phase 1
exit test needs the overnight run.

The row is tagged `smoke` so it never gets mistaken for a real number.

In [ ]:
import glob
ckpt = sorted(glob.glob("/kaggle/working/smoke/ckpt_*.pt"))[-1]
print(ckpt)
!python evaluation/eval.py --ckpt $ckpt --data /kaggle/input --out /kaggle/working/results \
    --limit 5 --notes smoke
!python evaluation/plots.py --csv /kaggle/working/smoke/train_log.csv \
    --out /kaggle/working/results/figures

In [ ]:
from IPython.display import Image as Show, display
import pandas as pd
display(pd.read_csv("/kaggle/working/results/benchmark.csv"))
for f in ["loss_curve.png", "psnr_curve.png", "qualitative.png"]:
    display(Show(f"/kaggle/working/results/figures/{f}"))

All green? Start `kaggle_train.ipynb` and leave it running.

Before closing the session, download `/kaggle/working/results/` and commit the
CSV and figures. They are the evidence the run happened, and `benchmark.csv` is
the file every later phase appends to.